<div class="freebsdLab-IntroCard">
  <div class="freebsdLab-IntroBrand">FreeBSD Laboratory</div>
  <h1 class="freebsdLab-IntroTitle">Autonomous AI Agent Subsystem</h1>
  <p class="freebsdLab-IntroTagline">Governed local model inference in disposable, isolated runtimes.</p>
  <p class="freebsdLab-IntroCopy">This notebook demonstrates the FreeBSD Laboratory autonomous agent controller architecture. Small instruction-tuned LLMs (such as Gemma 2B via <code>llama-cpp-python</code>) propose guest-level shell actions executed inside disposable FreeBSD runtimes while preserving the repository's strict trust boundaries.</p>
  <div class="freebsdLab-IntroNotice">
    <div class="freebsdLab-IntroNoticeIcon">i</div>
    <div>
      <div class="freebsdLab-IntroNoticeLabel">Architecture Invariant</div>
      <div class="freebsdLab-IntroNoticeText">The model has zero visibility into <code>runtime.sock</code>, ZFS datasets, SSH key paths, or host filesystem details. The unprivileged Agent Controller governs all execution, enforces output/timeout bounds, and emits hash-only evidence logs.</div>
    </div>
    <div class="freebsdLab-IntroNoticeMeta"><strong>Agent</strong>bhyve + jail</div>
  </div>
</div>

## 1. Architecture & Trust Hierarchy

```text
┌──────────────────────┐
│  Local LLM Engine    │
│  llama-cpp-python    │
│  GGUF model          │
└──────────┬───────────┘
           │ proposed action (COMMAND: | FINAL:)
           ▼
┌──────────────────────┐
│  Agent Controller    │ unprivileged
│  command policy      │ timeout & output bounds
│  evidence generation │ context token budget
└──────┬────────┬──────┘
       │        │
       ▼        ▼
 RuntimeClient  SSHTransport
 (runtime.sock) (freebsd@, per-runtime key)
       │        │
       ▼        ▼
 runtime-daemon Isolated Runtime (bhyve default, jail opt-in)
```

- **bhyve default**: Hardware virtualization provides the primary boundary against untrusted autonomous execution.
- **Unprivileged SSH**: Reuses `SSHTransport` with `freebsd` user and per-runtime Ed25519 keys (no root login).
- **Bounded I/O**: Stream-drained Popen execution with 4 KiB head + 4 KiB tail rolling buffers.

## 2. Action Protocol & Parsing

The model operates under a minimal two-action response grammar:
- `COMMAND: <single shell action>` — propose an action to execute in the guest
- `FINAL: <task result>` — declare the task complete with summary findings

The cell below loads the protocol definitions (using host imports or the embedded reference implementation).

In [ ]:
from dataclasses import dataclass
from typing import Union

try:
    from freebsd_laboratory.agent.types import Command, FinalAnswer, BoundedOutput, Observation
    from freebsd_laboratory.agent.model import parse_action
except ImportError:
    @dataclass(frozen=True)
    class Command:
        command: str

    @dataclass(frozen=True)
    class FinalAnswer:
        answer: str

    def parse_action(text: str) -> Union[Command, FinalAnswer]:
        stripped = text.strip()
        if not stripped:
            return FinalAnswer("Task complete (empty response).")
        for line in stripped.splitlines():
            line_clean = line.strip()
            if line_clean.upper().startswith("COMMAND:"):
                cmd = line_clean[len("COMMAND:"):].strip()
                if cmd:
                    return Command(cmd)
            elif line_clean.upper().startswith("FINAL:"):
                ans = line_clean[len("FINAL:"):].strip()
                return FinalAnswer(ans or stripped)
        first_line = stripped.splitlines()[0].strip()
        if len(stripped.splitlines()) == 1 and not first_line.startswith(("#", "//", "I ", "Let ")):
            return Command(first_line)
        return FinalAnswer(stripped)

print("[OK] Action protocol loaded.")

In [ ]:
# Test parsing different model response formats
sample_responses = [
    "COMMAND: sysctl hw.model hw.ncpu",
    "   COMMAND:   ifconfig -a   ",
    "I have analyzed the system.\nFINAL: FreeBSD 15.1 running on bhyve with active VNET networking.",
    "df -h /",
]

for resp in sample_responses:
    action = parse_action(resp)
    action_val = getattr(action, 'command', getattr(action, 'answer', ''))
    print(f"Raw input : {resp!r}")
    print(f"Parsed as : {type(action).__name__} -> {action_val}")
    print("-" * 60)

## 3. Structural Policy & Command Authorization

The controller enforces structural bounds (command byte length, step limits, session deadline). Because runtimes are disposable and network-isolated, destructive actions (e.g. `rm -rf /`) are contained inside the guest rather than fragilely denylisted.

In [ ]:
try:
    from freebsd_laboratory.agent.policy import AgentPolicy, Decision
except ImportError:
    @dataclass(frozen=True)
    class Decision:
        authorized: bool
        reason: str = ""

    class AgentPolicy:
        def __init__(self, max_command_bytes: int = 4096, max_steps: int = 16, max_runtime_seconds: float = 300):
            self.max_command_bytes = max_command_bytes
            self.max_steps = max_steps
            self.max_runtime_seconds = float(max_runtime_seconds)

        def authorize(self, command: str, step: int, elapsed: float) -> Decision:
            if not isinstance(command, str) or not command.strip():
                return Decision(False, "empty command")
            encoded_len = len(command.encode("utf-8"))
            if encoded_len > self.max_command_bytes:
                return Decision(False, f"command length ({encoded_len} bytes) exceeds limit")
            if step >= self.max_steps:
                return Decision(False, f"step limit ({self.max_steps}) reached")
            if elapsed >= self.max_runtime_seconds:
                return Decision(False, f"session deadline ({self.max_runtime_seconds}s) reached")
            return Decision(True)

print("[OK] AgentPolicy loaded.")

In [ ]:
policy = AgentPolicy(max_command_bytes=256, max_steps=5, max_runtime_seconds=60)

commands_to_test = [
    ("uname -a", 0, 1.2),
    ("echo " + "x" * 300, 1, 2.5),       # Exceeds byte cap
    ("rm -rf /tmp/scratch", 2, 4.0),     # Destructive but allowed in disposable runtime
    ("   ", 3, 5.0),                     # Empty command
    ("sysctl kern.ostype", 5, 8.0),      # Step limit reached
    ("uptime", 4, 65.0),                 # Session deadline exceeded
]

for cmd, step, elapsed in commands_to_test:
    decision = policy.authorize(cmd, step=step, elapsed=elapsed)
    status = "AUTHORIZED" if decision.authorized else f"REJECTED ({decision.reason})"
    print(f"Step {step} (t={elapsed:4.1f}s) | {cmd[:30]:<30} -> {status}")

## 4. Bounded Execution & Concurrent Stream Draining

`bounded_exec()` prevents controller memory exhaustion by draining stdout and stderr concurrently with reader threads, keeping the first 4 KiB (head) and last 4 KiB (rolling tail).

In [ ]:
import subprocess
import threading

try:
    from freebsd_laboratory.agent.bounded_exec import bounded_exec, BoundedOutput
except ImportError:
    @dataclass(frozen=True)
    class BoundedOutput:
        head: bytes
        tail: bytes
        total_bytes: int
        truncated: bool

    def bounded_exec(command, timeout=30.0, head_limit=4096, tail_limit=4096):
        proc = subprocess.Popen(
            command, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
        out_head, out_tail = bytearray(), bytearray()
        out_total = [0]
        
        def drain(stream, head, tail, total):
            while True:
                chunk = stream.read(4096)
                if not chunk: break
                total[0] += len(chunk)
                if len(head) < head_limit:
                    take = min(len(chunk), head_limit - len(head))
                    head.extend(chunk[:take])
                    chunk = chunk[take:]
                if chunk and tail_limit > 0:
                    tail.extend(chunk)
                    if len(tail) > tail_limit: tail[:] = tail[-tail_limit:]

        t_out = threading.Thread(target=drain, args=(proc.stdout, out_head, out_tail, out_total))
        t_out.start()
        proc.wait(timeout=timeout)
        t_out.join()
        
        trunc = out_total[0] > (head_limit + tail_limit)
        stdout_res = BoundedOutput(bytes(out_head), bytes(out_tail), out_total[0], trunc)
        stderr_res = BoundedOutput(b"", b"", 0, False)
        return proc.returncode, stdout_res, stderr_res

# Run a test command producing 50 KiB of output
code = "import sys; sys.stdout.write('START_MARKER\\n' + 'DATA_LINE\\n' * 5000 + 'END_MARKER\\n')"
exit_code, stdout_out, stderr_out = bounded_exec(
    [sys.executable, "-c", code],
    timeout=5.0,
    head_limit=256,
    tail_limit=256,
)

print(f"Exit status   : {exit_code}")
print(f"Total bytes   : {stdout_out.total_bytes} bytes")
print(f"Truncated?    : {stdout_out.truncated}")
print(f"Head snippet  : {stdout_out.head[:40]!r}...")
print(f"Tail snippet  : ...{stdout_out.tail[-40:]!r}")

## 5. Hash-Only Evidence Generation

The agent persists execution events into a standalone append-only JSONL log. To ensure privacy and prevent secret leakage, only SHA-256 digests and byte counters are stored on disk.

In [ ]:
import hashlib
import json
import os
import tempfile
from datetime import datetime, timezone
from pathlib import Path

try:
    from freebsd_laboratory.agent.evidence import AgentEvidenceLog, make_command_event
    from freebsd_laboratory.agent.types import Observation
except ImportError:
    @dataclass(frozen=True)
    class Observation:
        step: int
        command: str
        exit_status: int
        stdout: BoundedOutput
        stderr: BoundedOutput
        duration_ms: int

    def sha256_bytes(data: bytes) -> str:
        return hashlib.sha256(data).hexdigest()

    def make_command_event(session_id, runtime_id, runtime_type, obs):
        return {
            "event": "agent-command-complete",
            "session_id": session_id,
            "runtime_id": runtime_id,
            "runtime_type": runtime_type,
            "step": obs.step,
            "command_sha256": sha256_bytes(obs.command.encode("utf-8")),
            "exit_status": obs.exit_status,
            "stdout_sha256": sha256_bytes(obs.stdout.head + obs.stdout.tail),
            "stdout_bytes": obs.stdout.total_bytes,
            "stderr_sha256": sha256_bytes(obs.stderr.head + obs.stderr.tail),
            "stderr_bytes": obs.stderr.total_bytes,
            "duration_ms": obs.duration_ms,
            "truncated": obs.stdout.truncated or obs.stderr.truncated,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        }

with tempfile.TemporaryDirectory() as tmpdir:
    log_file = Path(tmpdir) / "agent_evidence.jsonl"
    obs = Observation(
        step=0,
        command="sysctl kern.ostype kern.osrelease",
        exit_status=0,
        stdout=BoundedOutput(head=b"FreeBSD 15.1\n", tail=b"", total_bytes=13, truncated=False),
        stderr=BoundedOutput(head=b"", tail=b"", total_bytes=0, truncated=False),
        duration_ms=25,
    )
    
    event = make_command_event("demo-session-01", "freebsd-lab-a123", "bhyve", obs)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(json.dumps(event, sort_keys=True) + "\n")
        
    line = log_file.read_text().strip()
    print("Persisted JSONL Event (SHA-256 hashes only):")
    print(json.dumps(json.loads(line), indent=2))
    
    # Verify invariant: no raw command string stored
    assert "sysctl" not in line
    print("\n[PASSED] Invariant verified: Raw command string is not present in persistent evidence.")

## 6. End-to-End Agent Orchestration Simulation

We run a live multi-step agent session executing real inspection commands (`uname`, `sysctl`, `uptime`) inside the current runtime.

In [ ]:
import platform
import subprocess
import time

goal = "Audit kernel identity, active CPU count, and system uptime"
plan_commands = [
    "uname -s -r -p 2>/dev/null || uname -a",
    "sysctl -n hw.ncpu 2>/dev/null || echo '1'",
    "uptime",
]

print(f"Target Goal: {goal!r}\n")
observations = []

for step, cmd in enumerate(plan_commands):
    print(f"[Step {step}] Proposing: COMMAND: {cmd}")
    start = time.monotonic()
    res = subprocess.run(["sh", "-c", cmd], capture_output=True, text=True)
    duration_ms = int((time.monotonic() - start) * 1000)
    out = res.stdout.strip() or res.stderr.strip() or "(empty)"
    
    print(f"  ↳ Exit: {res.returncode} ({duration_ms}ms)")
    print(f"  ↳ Output:\n{out}\n")
    observations.append(out)

tokens = observations[0].split() if observations and observations[0] else []
sys_name = tokens[0] if len(tokens) > 0 else "FreeBSD"
sys_rel = tokens[1] if len(tokens) > 1 else ""
cpu_count = observations[1].strip() if len(observations) > 1 and observations[1] else "1"

final_summary = f"FINAL: Host identified as {sys_name} {sys_rel} with {cpu_count} CPU(s) active."
print("=" * 60)
print(f"Agent Result: {final_summary}")

## 7. CLI Usage Reference

In production on the FreeBSD host, autonomous agents are executed using the `freebsd-lab-agent` CLI:

```sh
# Run an autonomous agent inside a strong bhyve VM boundary (default):
freebsd-lab-agent "Analyze /var/log/messages and report any kernel panic indicators" \
    --model /models/gemma-2b-it.gguf \
    --mode bhyve \
    --max-steps 16 \
    --max-runtime 300 \
    --evidence-dir .freebsd-lab/agent-evidence

# Run in a lightweight VNET jail for fast controlled benchmarks:
freebsd-lab-agent "Check installed package versions and audit report" \
    --model /models/gemma-2b-it.gguf \
    --mode jail
```